**Система:** театральная касса  
**Технология взаимодействия:** XML-RPC  
**Формат хранения записей:** `dict`  
**Поля:** номер записи, театр, название спектакля, стоимость билета  
**Операции:** добавление, редактирование, подсчет записей, справка и отчет по минимальной стоимости билета

Проект содержит серверную часть, клиентскую часть и локальную проверку XML-RPC-взаимодействия.

## 0. Импорт библиотек

In [ ]:
import threading
import time
import xmlrpc.client
import xmlrpc.server

## 1. Серверная логика

In [ ]:
class TheaterTicketSystem:
    def __init__(self):
        self.records = {}
        self.next_id = 1
        self._initialize_sample_data()

    def _initialize_sample_data(self):
        sample_data = [
            {
                "theater": "Большой театр",
                "play_name": "Лебединое озеро",
                "ticket_price": 2800
            },
            {
                "theater": "МХТ им. Чехова",
                "play_name": "Три сестры",
                "ticket_price": 2000
            },
            {
                "theater": "Ленком",
                "play_name": "Красавица Ангара",
                "ticket_price": 2200
            },
            {
                "theater": "Современник",
                "play_name": "Гроза",
                "ticket_price": 1700
            },
            {
                "theater": "Большой театр",
                "play_name": "Щелкунчик",
                "ticket_price": 3000
            },
            {
                "theater": "Театр Вахтангова",
                "play_name": "Принцесса Турандот",
                "ticket_price": 1900
            },
            {
                "theater": "Мариинский театр",
                "play_name": "Борис Годунов",
                "ticket_price": 3500
            },
        ]

        for item in sample_data:
            self.add_record(
                item["theater"],
                item["play_name"],
                item["ticket_price"]
            )

    def add_record(self, theater, play_name, ticket_price):
        try:
            price = float(ticket_price)

            record_id = self.next_id
            self.records[record_id] = {
                "theater": theater,
                "play_name": play_name,
                "ticket_price": price
            }

            self.next_id += 1
            return f"Запись успешно добавлена с ID: {record_id}"

        except ValueError:
            return "Ошибка: стоимость билета должна быть числом"

    def update_record(self, record_id, theater, play_name, ticket_price):
        try:
            record_id = int(record_id)
            price = float(ticket_price)

            if record_id not in self.records:
                return f"Ошибка: запись с ID {record_id} не найдена"

            self.records[record_id] = {
                "theater": theater,
                "play_name": play_name,
                "ticket_price": price
            }

            return f"Запись с ID {record_id} успешно обновлена"

        except ValueError:
            return "Ошибка: ID и стоимость должны быть числами"

    def get_record_count(self):
        return len(self.records)

    def get_min_prices_report(self):
        theater_min_prices = {}

        for record in self.records.values():
            theater = record["theater"]
            price = record["ticket_price"]

            if (
                theater not in theater_min_prices
                or price < theater_min_prices[theater]
            ):
                theater_min_prices[theater] = price

        lines = ["Минимальная стоимость билета по театрам"]

        for theater, price in sorted(theater_min_prices.items()):
            lines.append(f"{theater}: {price:.2f} руб.")

        return "\n".join(lines)

    def get_all_records(self):
        return self.records

    def get_help(self):
        return (
            "Команды:\\n"
            "add — добавить запись\\n"
            "update — изменить запись\\n"
            "count — количество записей\\n"
            "report — минимальные цены по театрам\\n"
            "help — справка\\n"
            "exit — выход"
        )

## 2. Проверка серверной логики

In [ ]:
system = TheaterTicketSystem()

print("Количество записей:", system.get_record_count())
print()
print(system.get_min_prices_report())

## 3. XML-RPC сервер

In [ ]:
HOST = "127.0.0.1"
PORT = 8000

rpc_server = xmlrpc.server.SimpleXMLRPCServer(
    (HOST, PORT),
    allow_none=True,
    logRequests=False
)

rpc_server.register_instance(TheaterTicketSystem())

server_thread = threading.Thread(
    target=rpc_server.serve_forever,
    daemon=True
)

server_thread.start()

print(f"XML-RPC сервер запущен: http://{HOST}:{PORT}")

## 4. XML-RPC клиент

In [ ]:
client = xmlrpc.client.ServerProxy(
    f"http://{HOST}:{PORT}",
    allow_none=True
)

print(client.get_help())

## 5. Работа с записями через клиент

In [ ]:
print(client.add_record(
    "Театр на Таганке",
    "Мастер и Маргарита",
    2400
))

print()
print("Количество записей:", client.get_record_count())

print()
print(client.update_record(
    1,
    "Большой театр",
    "Лебединое озеро",
    2750
))

## 6. Отчет по минимальной стоимости

In [ ]:
print(client.get_min_prices_report())

## 7. Содержимое словаря

In [ ]:
records = client.get_all_records()

for record_id, record in records.items():
    print(
        f"ID {record_id}: "
        f"{record['theater']} — "
        f"{record['play_name']} — "
        f"{record['ticket_price']:.2f} руб."
    )

## 8. Остановка сервера

In [ ]:
rpc_server.shutdown()
rpc_server.server_close()
print("XML-RPC сервер остановлен.")

## Итог

In [ ]:
print("Доступные операции:")
print("add_record")
print("update_record")
print("get_record_count")
print("get_min_prices_report")
print("get_all_records")
print("get_help")